# 1.6 - K-Means Clustering

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook builds K-Means three ways: the raw 4-step loop by hand, the production scikit-learn version with a proper way to pick K, and a back-of-the-envelope model of how vector databases like FAISS use K-Means to search a million documents while touching only a sliver of them. The goal is to see the *same* algorithm move from a textbook loop to the thing that makes semantic search fast.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the libraries this lesson leans on: NumPy for the math, scikit-learn for the production clustering and metrics, and `datasets` (available if you extend the exercises to real embeddings).

In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q scikit-learn numpy datasets


**Explanation**

A one-line environment prep cell that installs the packages the rest of the notebook imports. Nothing here computes anything.

**How the code works - step by step**
- **`!pip install -q ...`** - installs `scikit-learn`, `numpy`, and `datasets`; the `-q` flag suppresses the pip logs.

**In one line:** get the toolbox on the machine before any code runs.

**What you'll see:** (no output - environment prep)

## 1 - K-Means from scratch, the 4-step loop

K-Means has exactly one idea repeated until it stops changing: put each point with its nearest centroid, then move each centroid to the average of its points. Writing the loop yourself is the fastest way to see that there's no magic - it's `argmin` plus `mean` in a `while` loop. This is the identical algorithm FAISS runs to build a search index, just on toy data here.

In [ ]:
# =============================================
# K-Means from scratch — the 4-step loop
# Same algorithm FAISS uses at scale
# =============================================

import numpy as np
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

def kmeans_scratch(X, k, max_iters=100):
    # Step 1: Random initialization
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].copy()

    for iteration in range(max_iters):
        # Step 2: Assign each point to nearest centroid
        distances = np.linalg.norm(X[:, None] - centroids, axis=2)
        labels = distances.argmin(axis=1)

        # Step 3: Update centroids to cluster means
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])

        # Step 4: Check convergence
        if np.allclose(centroids, new_centroids):
            print(f"Converged at iteration {iteration}")
            break
        centroids = new_centroids

    return labels, centroids

labels, centroids = kmeans_scratch(X, k=3)
print(f"Cluster sizes: {np.bincount(labels)}")
print(f"Centroids:\n{centroids.round(2)}")


**Explanation**

This cell generates 300 points in 3 natural blobs, then hand-codes the full K-Means loop as a function. It's a from-scratch implementation, not a library call - every step (init, assign, update, check-convergence) is visible NumPy. Read it as: seed some centroids, then alternate assign-and-average until the centroids stop moving.

**How the code works - step by step**
- **`make_blobs(...)`** - creates 300 2-D points clustered around 3 centers, so we know the right answer is k=3.
- **Step 1 - init** - `np.random.choice` picks `k` random points from the data to use as starting centroids.
- **Step 2 - assign** - `np.linalg.norm(X[:, None] - centroids, axis=2)` computes every point-to-centroid distance at once; `argmin(axis=1)` labels each point with its nearest centroid.
- **Step 3 - update** - for each cluster, recompute the centroid as the mean of the points currently assigned to it.
- **Step 4 - converge** - `np.allclose` checks whether the centroids barely moved; if so, print the iteration and stop. Otherwise keep the new centroids and loop.

**In one line:** assign to nearest -> move to the mean -> repeat until nothing moves.

**What you'll see:** A `Converged at iteration N` line (it settles in just a few passes), then the three cluster sizes via `np.bincount` (roughly 100 each) and the final 3 centroid coordinates rounded to 2 decimals.

## 2 - Picking K with inertia and the silhouette score

The hard part of K-Means isn't running it - it's choosing K, because K-Means will happily split data into any number of clusters you ask for. This cell sweeps K from 2 to 7 and prints two diagnostics side by side so you can see which K the data actually wants.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs
import numpy as np

X, _ = make_blobs(n_samples=500, centers=4, random_state=42)

print(f"{'K':>3} {'Inertia':>10} {'Silhouette':>11} {'Verdict'}")
print("-" * 42)
for k in range(2, 8):
    km = KMeans(k, random_state=42, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    verdict = " ← optimal" if k == 4 else ""
    print(f"  {k:>2} {km.inertia_:>10.0f} {sil:>10.3f}{verdict}")


**Explanation**

A measurement harness, not a new algorithm. It runs scikit-learn's production `KMeans` at several values of K and scores each with two metrics: inertia (within-cluster tightness, always falls as K grows - the 'elbow') and the silhouette score (cluster separation, which peaks at the right K). The point is to contrast a metric that always improves against one that has a real maximum.

**How the code works - step by step**
- **`make_blobs(centers=4)`** - builds 500 points in 4 true clusters, so the honest answer is K=4.
- **`KMeans(k, n_init=10)`** - scikit-learn's version; `n_init=10` restarts from 10 random seeds and keeps the best, avoiding bad initializations.
- **`km.inertia_`** - sum of squared distances to each point's centroid; monotonically decreasing, so you look for the 'elbow' not the minimum.
- **`silhouette_score(...)`** - measures how much closer points sit to their own cluster than the next-nearest; higher is better and it genuinely peaks.
- **`verdict`** - hard-codes the ` ← optimal` marker on the K=4 row for emphasis.

**In one line:** inertia always drops, silhouette peaks - trust the peak.

**What you'll see:** A 6-row table (K = 2..7) with columns Inertia, Silhouette, and Verdict. Inertia shrinks steadily as K rises, while the silhouette is highest at K=4, which carries the `← optimal` tag.

## 3 - Why FAISS clusters a million documents

This is the payoff: the same K-Means you just built is how vector databases make semantic search fast. FAISS's IVF index clusters all your embeddings once at build time, then at query time only searches the few clusters nearest to the query instead of all million vectors. This cell does the arithmetic on that trade.

In [ ]:
import numpy as np

# Simulate FAISS IVF logic (conceptual)
n_docs = 1_000_000
nlist = 1024   # K-Means clusters at index time
nprobe = 16    # clusters searched at query time

docs_per_cluster = n_docs // nlist
docs_searched = nprobe * docs_per_cluster
speedup = n_docs / docs_searched
pct_searched = (docs_searched / n_docs) * 100

print(f"Documents:          {n_docs:>10,}")
print(f"K-Means clusters:   {nlist:>10,}")
print(f"Clusters probed:    {nprobe:>10,}")
print(f"Docs searched:      {docs_searched:>10,}")
print(f"% of database:      {pct_searched:>9.1f}%")
print(f"Speedup:            {speedup:>9.0f}x")
print(f"Recall:             >95%")


**Explanation**

A pure arithmetic simulation - no clustering actually runs, it models the cost of an IVF (inverted-file) index conceptually. It takes an index configured with 1024 clusters, searches only 16 of them per query, and computes how much of the database that skips and the resulting speedup. Read it as the math that justifies clustering-based retrieval.

**How the code works - step by step**
- **`nlist = 1024`** - number of K-Means clusters built over the 1M documents at index time.
- **`nprobe = 16`** - number of nearest clusters actually scanned per query (the recall/speed knob).
- **`docs_per_cluster`** - 1M / 1024, the average bucket size.
- **`docs_searched`** - `nprobe * docs_per_cluster`, how many vectors a query actually touches.
- **`speedup` / `pct_searched`** - the ratio of full-scan to actual work, and what fraction of the DB that represents.

**In one line:** cluster once, probe a handful of buckets, and search ~1.6% of the data for >95% recall.

**What you'll see:** A formatted report: 1,000,000 documents, 1,024 clusters, 16 probed, about 15,616 docs searched (~1.6% of the database), a ~64x speedup, and a note that recall stays above 95%.

You built K-Means by hand (assign to nearest centroid, move centroids to the mean, repeat until nothing moves), then let scikit-learn do it while you used inertia and the silhouette score to argue for the right K, then modeled why that same clustering is what lets a vector index search 1M docs by scanning ~1.6% of them. The through-line: clustering isn't just an analysis tool, it's the indexing trick underneath fast retrieval. Next up in Module 8 (RAG Systems), that FAISS IVF math stops being a simulation and becomes the actual `nlist`/`nprobe` knobs you tune on a real index.